<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/06-dictionaries_pandas/06_intro_to_dictionaries_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: Dictionaries and pandas

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0), and
- [pandas documentation](https://pandas.pydata.org/docs/)

```
ASK  ->  [ GET ]  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
```

So far we have used Python (Module 3), NumPy (Module 4), and seaborn (Module 5).

NumPy gave us fast arrays of numbers, but real datasets are **tables**: many rows,
several columns, and a mix of text and numbers in each row.

This module gives us two tools that carry us through the **GET** stage and into **EXPLORE**: the Python
*dictionary* (a labeled lookup) and the pandas *DataFrame* (a spreadsheet in code).

## Learning objectives

1. Store labeled data in a **dictionary** and look values up by key.
2. Add, modify, delete, and iterate over key-value pairs.
3. Recognize a *record* as a dictionary and a *table* as a collection of records.
4. Build a pandas **DataFrame** from dictionaries and read one from a CSV file.
5. Inspect, select, filter, and sort a DataFrame.
6. Create new columns and compute descriptive statistics.

Our running dataset for this module is the **most-streamed songs on Spotify**. It is small
enough to read at a glance and familiar enough to reason about.

---
## 1. Dictionaries
---

### 1.1 The lookup problem

Suppose the campus radio station is tracking how many times some popular songs have been streamed (in billions). A first instinct, using what we know from Module 3, is two parallel lists:

`titles  = ['Blinding Lights', 'Shape of You', 'Riptide']`

`streams = [5.3, 4.8, 3.5]`

Simple question: *How many streams does 'Riptide' have?*
With lists, we first have to find where the title lives, then read the matching position in the other list.

In [ ]:
# RUN-TOGETHER

titles  = ['Blinding Lights', 'Shape of You', 'Riptide']
streams = [5.3, 4.8, 3.5]

# To look up Riptide we must find its position first, then index
# the OTHER list.
idx = titles.index('Riptide')     # position of the title
print('position:', idx)
print('streams (billions):', streams[idx])

position: 2
streams (billions): 3.5


This works, but it is fragile:

- The two lists must stay in the **same order** forever. If we sort one and forget the other,
  every lookup silently returns the wrong song.
- Looking something up takes **two steps** (find the position, then index).

We want to ask for a value by its name, in one step. That is exactly what a **dictionary** does.

### 1.2 Dictionaries: lookup by key

**Definition.** A *dictionary* is an unordered collection of **key-value pairs**, written in
curly braces `{ }`. Each **key** points to a **value**, and we retrieve a value by writing the key in square brackets: `d[key]`.

- Keys must be unique and immutable (strings, ints, floats, booleans). Text keys are the most common.
- Values can be anything: numbers, strings, lists, or even other dictionaries.

Instead of two lists that must stay aligned, we bind each title to its streams figure directly.

In [ ]:
# RUN-TOGETHER
# Curly braces, and 'key : value' pairs separated by commas.
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Riptide': 3.5}

# One-step lookup by name -- no positions, no second list.
print(song_streams['Riptide'])

# The whole dictionary:
print(song_streams)
print(type(song_streams))

3.5
{'Blinding Lights': 5.3, 'Shape of You': 4.8, 'Riptide': 3.5}
<class 'dict'>


### Activity 1

The station wants a quick "how far ahead is the leader?" stat.

A. Using two lookups and subtraction, print how many billion streams `'Blinding Lights'` is ahead of `'Riptide'`.

B. What does Python do if you ask for a title that is *not* a key, such as `song_streams['Levitating']`?

(We fix this safely in the next section.)

In [ ]:
# Activity code:
# a. How far ahead is Blinding Lights over Riptide?
gap = song_streams[____] - song_streams[____]
print('Blinding Lights leads Riptide by', round(gap, 1), 'billion streams')

# b. PREDICT (comment only, do not run a missing-key lookup yet):
#    song_streams['Levitating'] would do what?
#    my guess: ____

### 1.3 Adding, modifying, and deleting pairs

A dictionary is *changeable*. We use the same square-bracket syntax to store a value:

- **Add** a new pair: `d[new_key] = value`
- **Modify** an existing value: `d[existing_key] = new_value` (same syntax, an existing key)
- **Check membership**: `key in d` returns `True`/`False`
- **Delete** a pair: `del d[key]`

Real data arrives incrementally and gets corrected. Adding, fixing, and removing
entries is the day-to-day reality of maintaining a dataset.

In [ ]:
# FILL-IN

# ADD a song the station just started tracking.
song_streams['Die With a Smile'] = 3.5
print('after adding Die With a Smile:', song_streams)

# MEMBERSHIP test: is a key present?
print("'Die With a Smile' in song_streams? ->", 'Die With a Smile' in song_streams)
print("'Levitating' in song_streams?       ->", 'Levitating' in song_streams)

# MODIFY: streams keep climbing, so bump Die With a Smile up a little.
song_streams['Die With a Smile'] = 3.6
print('after updating Die With a Smile:', song_streams['Die With a Smile'])

# DELETE: the station stops tracking Shape of You.
del song_streams['Shape of You']
print('after deleting Shape of You:', song_streams)

after adding Die With a Smile: {'Blinding Lights': 5.3, 'Shape of You': 4.8, 'Riptide': 3.5, 'Die With a Smile': 3.5}
'Die With a Smile' in song_streams? -> True
'Levitating' in song_streams?       -> False
after updating Die With a Smile: 3.6
after deleting Shape of You: {'Blinding Lights': 5.3, 'Riptide': 3.5, 'Die With a Smile': 3.6}


### Activity 2

Real datasets get *corrected*, not just overwritten. Working from the current `song_streams`:

a. **Correction without hard-coding a number.** The station realizes `'Blinding Lights'` was over-counted by `0.4`. Lower it using its own current value (read the value, subtract, store it back).

b. **Add** `'Heat Waves'` at `3.7`.

c. **Safe lookup.** Complete an `if`/`else` that prints a song's streams *only if* the title is present, and otherwise prints "not tracked". Test it on `'Yellow'` (which is absent).

In [ ]:
# Activity 2
# a. read-modify-write: lower Blinding Lights by 0.4 using its CURRENT value (no typed number)
song_streams['Blinding Lights'] = song_streams[____] ____ 0.4

# b. add Heat Waves at 3.7
song_streams[____] = ____

# c. safe lookup guarded by `in`
title = 'Yellow'
if title ____ song_streams:
    print(title, '->', song_streams[title])
else:
    print(title, 'is not tracked')

print(song_streams)

### 1.4 Iterating over a dictionary

Often we want to *walk through every pair* to print a report, add things up, or transform the data. Three views help:

- `d.keys()`   -> the keys
- `d.values()` -> the values
- `d.items()`  -> the (key, value) pairs

A `for` loop over `d.items()` hands us the key and value together on each pass.

In [ ]:
# RUN-TOGETHER
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Heat Waves': 3.7,
                'Riptide': 3.5,
                'Die With a Smile': 3.5}

# Loop over key/value pairs with .items()
for title, plays in song_streams.items():
    print(title, 'has about', plays, 'billion streams')

Blinding Lights has about 5.3 billion streams
Shape of You has about 4.8 billion streams
Heat Waves has about 3.7 billion streams
Riptide has about 3.5 billion streams
Die With a Smile has about 3.5 billion streams


Because we can visit every value, we can also *summarize* the dictionary. But summarizing takes looping through all items in dictionary.

Before running the code below, lets write the steps by hand:

Step 0: Initially, total = 0, no top_title, and top_plays = 0

Step 1: First item in dictionary

title = 'Blinding Lights', plays = 5.3
total = 0 + 5.3 = 5.3
since 5.3 > top_plays = 0, set top_plays = 5.3 and top_title = 'Blinding Light'
move to next step

Step 2: Next item in Dictionary

title = 'Shape of You', plays = 4.8
total = 5.3 + 4.8 = 10.1
since 4.8 is not > top_plays = 5.3, move to next step

Etc....total keeps increasing and there is not another plays that will be larger than 5.3

In [ ]:
## Example: Total the streams, and find the most-streamed song
# by walking through the pairs.
# FILL-IN-LIVE

# initial state
total = 0
top_title = None
top_plays = 0

# loop through dictionary
for title, plays in song_streams.items():
    total = total + plays               # running sum of plays
    if plays > top_plays:               # track the largest so far
        top_plays = plays
        top_title = title

print('total streams (billions):', round(total, 1))
print('most-streamed:', top_title, '(', top_plays, 'billion )')

total streams (billions): 20.8
most-streamed: Blinding Lights ( 5.3 billion )


### 1.5 A record is a dictionary

So far each value has been a single number. But a dictionary can describe one whole thing by using *different fields as keys*. This is called a **record**.

**Definition.** A *record* is a dictionary whose keys are **field names** (like `title`, `year`)
and whose values are that one item's data. A record is the natural stand-in for one row of a table.

In [ ]:
# RUN-TOGETHER
# One song described as a record: several fields, mixed value types.
blinding_lights = {'title': 'Blinding Lights',
                   'artist': 'The Weeknd',
                   'year': 2019,
                   'genre': 'Synth-pop',
                   'streams_billions': 5.3}

# Read individual fields by key.
print(blinding_lights['title'], 'was released in', blinding_lights['year'])
print('genre:', blinding_lights['genre'])

Blinding Lights was released in 2019
genre: Synth-pop


If one record is a row, then a list of records is a **table** (or datatable). Here are three songs, each a record:

In [ ]:
# RUN-TOGETHER
# A list of dictionaries -- each dictionary is one row.
song_records = [
    {'title': 'Blinding Lights', 'artist': 'The Weeknd', 'year': 2019, 'streams_billions': 5.3},
    {'title': 'Riptide',         'artist': 'Vance Joy',  'year': 2013, 'streams_billions': 3.5},
    {'title': 'Bohemian Rhapsody','artist': 'Queen',     'year': 1975, 'streams_billions': 3.1},
]

# Reach into the second record (index 1), then the 'artist' field.
print(song_records[1]['title'], 'is by', song_records[1]['artist'])

Riptide is by Vance Joy


### Activity 4

Here is a short list of records. Print the `title` of every song released in *2015 or later*.


In [ ]:
# Activity 4
catalog = [
    {'title': 'Levitating', 'artist': 'Dua Lipa',    'year': 2020, 'streams_billions': 2.6},
    {'title': 'Yellow',     'artist': 'Coldplay',    'year': 2000, 'streams_billions': 3.6},
    {'title': 'Flowers',    'artist': 'Miley Cyrus', 'year': 2023, 'streams_billions': 2.8},
    {'title': 'Creep',      'artist': 'Radiohead',   'year': 1992, 'streams_billions': 2.7},
]

# Titles released in 2015 or later
for song in catalog:
    if song[____] >= 2015:
        print(song[____])

Levitating
Flowers


### Lists vs. Dictionaries: which one?

| | Indexed by | Order matters? | Good for |
| --- | --- | --- | --- |
| **List** | position (`0, 1, 2, ...`) | yes | a sequence of values (a column) |
| **Dictionary** | unique key | no | a fast lookup table; a record with named fields |

Rule of thumb: if you find yourself keeping two lists *in sync* so that position `i` in one
matches position `i` in another, you probably want a dictionary (or, very soon, a DataFrame).

---
## 2. From Dictionaries to pandas DataFrames
---

### 2.1 Why pandas?

A list of records is a table *in spirit*, but doing table work with raw dictionaries
(filtering rows, averaging a column) quickly gets clumsy. **pandas** is the standard Python
library for tabular data.

**Definition.** A *pandas DataFrame* is a two-dimensional table of data.
- Each **row** is one observation (here, one song).
- Each **column** is one feature (title, artist, year, ...), and has a label.
- It is built on top of NumPy, so column math is fast and element-wise, just like Module 4.

By convention we import it as `pd`.

In [ ]:
# RUN-TOGETHER
import pandas as pd

**Note:** Although we did not talk about it at the time, the Palmer Penguin dataset was imported as a panda.

### 2.2 Building a DataFrame from a dictionary

The most common pattern is a dictionary of columns: each **key** is a column name, and each **value** is a list holding that column's data, top to bottom.

In [ ]:
# RUN-TOGETHER
# A dictionary of columns: key = column name, value = list of column data.
data = {
    'title':  ['Blinding Lights', 'Shape of You', 'Riptide', 'Die With a Smile', 'Bohemian Rhapsody'],
    'artist': ['The Weeknd', 'Ed Sheeran', 'Vance Joy', 'Lady Gaga & Bruno Mars', 'Queen'],
    'year':   [2019, 2017, 2013, 2024, 1975],
    'streams_billions': [5.3, 4.8, 3.5, 3.5, 3.1],
}

songs_small = pd.DataFrame(data)
print(songs_small)

               title                  artist  year  streams_billions
0    Blinding Lights              The Weeknd  2019               5.3
1       Shape of You              Ed Sheeran  2017               4.8
2            Riptide               Vance Joy  2013               3.5
3   Die With a Smile  Lady Gaga & Bruno Mars  2024               3.5
4  Bohemian Rhapsody                   Queen  1975               3.1


### 2.3 Inspecting adataset

Let's load the station's full list of most-streamed songs. We build it once as a dictionary of columns, then turn it into a DataFrame called `songs`.

In [ ]:
# RUN-TOGETHER
# The full dataset (a dictionary of columns).
songs_data = {
    'title': ['Blinding Lights', 'Shape of You', 'Perfect', 'Believer', 'Heat Waves',
              'Yellow', 'Riptide', 'Die With a Smile', 'Take Me to Church', 'Die For You',
              'Bohemian Rhapsody', 'Mr. Brightside'],
    'artist': ['The Weeknd', 'Ed Sheeran', 'Ed Sheeran', 'Imagine Dragons', 'Glass Animals',
               'Coldplay', 'Vance Joy', 'Lady Gaga & Bruno Mars', 'Hozier', 'The Weeknd',
               'Queen', 'The Killers'],
    'year': [2019, 2017, 2017, 2017, 2020, 2000, 2013, 2024, 2013, 2016, 1975, 2004],
    'genre': ['Synth-pop', 'Pop', 'Pop', 'Pop rock', 'Indie pop', 'Alternative rock',
              'Indie folk', 'Pop', 'Indie rock', 'R&B', 'Rock', 'Rock'],
    'streams_billions': [5.3, 4.8, 3.9, 3.8, 3.7, 3.6, 3.5, 3.5, 3.4, 3.2, 3.1, 3.0],
}

songs = pd.DataFrame(songs_data)
songs

,title,artist,year,genre,streams_billions
0,Blinding Lights,The Weeknd,2019,Synth-pop,5.3
1,Shape of You,Ed Sheeran,2017,Pop,4.8
2,Perfect,Ed Sheeran,2017,Pop,3.9
3,Believer,Imagine Dragons,2017,Pop rock,3.8
4,Heat Waves,Glass Animals,2020,Indie pop,3.7
5,Yellow,Coldplay,2000,Alternative rock,3.6
6,Riptide,Vance Joy,2013,Indie folk,3.5
7,Die With a Smile,Lady Gaga & Bruno Mars,2024,Pop,3.5
8,Take Me to Church,Hozier,2013,Indie rock,3.4
9,Die For You,The Weeknd,2016,R&B,3.2


When you meet a table, look before you leap. These are the first commands to run on any new DataFrame:

- `df.head(n)`  -> the first `n` rows (default 5)
- `df.shape`    -> `(rows, columns)` -- an **attribute**, so no parentheses (recall Module 4)
- `df.columns`  -> the column labels
- `df.dtypes`   -> the type stored in each column
- `df.info()`   -> a compact summary: columns, non-null counts, dtypes
- `df.describe()` -> descriptive statistics for the numeric columns

In [ ]:
# RUN-TOGETHER
print('shape (rows, cols):', songs.shape)     # attribute: no ()
print()
print('column labels:', list(songs.columns))
print()
print('dtypes:')
print(songs.dtypes)

shape (rows, cols): (12, 5)

column labels: ['title', 'artist', 'year', 'genre', 'streams_billions']

dtypes:
title                   str
artist                  str
year                  int64
genre                   str
streams_billions    float64
dtype: object


In [ ]:
# RUN-TOGETHER
# OR we can run
songs.info()    # structure + non-null counts (great for spotting missing data)

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             12 non-null     str    
 1   artist            12 non-null     str    
 2   year              12 non-null     int64  
 3   genre             12 non-null     str    
 4   streams_billions  12 non-null     float64
dtypes: float64(1), int64(1), str(3)
memory usage: 612.0 bytes


In [ ]:
songs.describe()

,year,streams_billions
count,12.000000,12.000000
mean,2011.250000,3.733333
std,13.212425,0.680018
min,1975.000000,3.000000
25%,2010.750000,3.350000
50%,2016.500000,3.550000
75%,2017.500000,3.825000
max,2024.000000,5.300000


### 2.4 Reading data from a CSV file

Typing a dataset by hand is fine for a class demo, but in practice data lives in files. The most common is a **CSV** (comma-separated values): a plain-text table where columns are separated by
commas and each line is a row.

We rarely build the CSV ourselves, but doing the round trip once shows exactly what `read_csv` reads back. We save `songs` to a file, then read it into a fresh DataFrame.

In [ ]:
# RUN-TOGETHER
# Save our DataFrame to a CSV file. index=False keeps pandas from writing the row numbers.
songs.to_csv('songs.csv', index=False)

# Read it back. In real projects you START here, with a file someone handed you.
songs = pd.read_csv('songs.csv')

# ALWAYS check that it loaded the way you expected.
songs.head()

,title,artist,year,genre,streams_billions
0,Blinding Lights,The Weeknd,2019,Synth-pop,5.3
1,Shape of You,Ed Sheeran,2017,Pop,4.8
2,Perfect,Ed Sheeran,2017,Pop,3.9
3,Believer,Imagine Dragons,2017,Pop rock,3.8
4,Heat Waves,Glass Animals,2020,Indie pop,3.7


---
## 3. Exploring a DataFrame
---

With one clean table we can select, filter, sort, derive, and summarize. These are core moves of the EXPLORE stage.

### 3.1 Selecting columns

- **Single brackets** with one column name -> a Series (a labeled 1-D column).
- **Double brackets** with a list of names -> a DataFrame (a table, even for one column).

Think of it as: *"one thing in the brackets, get a column; a list in the brackets, get a table."*

In [ ]:
# RUN-TOGETHER
# Single brackets -> Series
titles = songs['title']
print(type(titles))     # pandas Series (a labeled 1-D array)
print(titles.head(3))

print()
# Double brackets -> DataFrame (note the list inside)
subset = songs[['title', 'streams_billions']]
print(type(subset))     # pandas DataFrame
subset.head(3)

<class 'pandas.Series'>
0    Blinding Lights
1       Shape of You
2            Perfect
Name: title, dtype: str

<class 'pandas.DataFrame'>


,title,streams_billions
0,Blinding Lights,5.3
1,Shape of You,4.8
2,Perfect,3.9


### 3.2 Selecting rows: `.iloc` and `.loc`

Two row selectors, mirroring the difference between *position* and *label*:

- `df.iloc[i]`  -> row by integer position (like list indexing; 0-based).
- `df.loc[label]` -> row by index label.

By default a DataFrame's index is `0, 1, 2, ...`, so `loc` and `iloc` look alike. They diverge
once we set a meaningful index -- here, the song's title.


In [ ]:
# RUN-TOGETHER
# Position-based: the first row, and rows 0..2.
print('first row (iloc[0]):')
print(songs.iloc[0])
print()
print('first three rows (iloc[0:3]):')
print(songs.iloc[0:3][['title', 'streams_billions']])

# Suppose the integer index is replaced with the song title
songs_by_title = songs.set_index('title')
print(songs_by_title.head())

# Same row, two ways: position 0 and the label sitting at position 0
#print(songs_by_title.iloc[0]['streams_billions'])
#print(songs_by_title.loc[____]['streams_billions'])   # use the LABEL

first row (iloc[0]):
title               Blinding Lights
artist                   The Weeknd
year                           2019
genre                     Synth-pop
streams_billions                5.3
Name: 0, dtype: object

first three rows (iloc[0:3]):
             title  streams_billions
0  Blinding Lights               5.3
1     Shape of You               4.8
2          Perfect               3.9
                          artist  year      genre  streams_billions
title                                                              
Blinding Lights       The Weeknd  2019  Synth-pop               5.3
Shape of You          Ed Sheeran  2017        Pop               4.8
Perfect               Ed Sheeran  2017        Pop               3.9
Believer         Imagine Dragons  2017   Pop rock               3.8
Heat Waves         Glass Animals  2020  Indie pop               3.7


### 3.3 Filtering rows with boolean masks

This is the same idea as NumPy boolean masks from Module 4, now on a whole table. A comparison
on a column produces a column of `True`/`False`; putting that mask in the brackets keeps only the
`True` rows.

- Combine conditions with `&` (and) and `|` (or), and wrap each condition in parentheses.

In [ ]:
# RUN-TOGETHER
# Step 1: a comparison makes a boolean column.
mask = songs['streams_billions'] > 4
print(mask.head())

print()
# Step 2: use the mask to keep matching rows.
mega = songs[mask]
print('songs over 4 billion streams:', mega.shape[0])
mega[['title', 'streams_billions']]

0     True
1     True
2    False
3    False
4    False
Name: streams_billions, dtype: bool

songs over 4 billion streams: 2


,title,streams_billions
0,Blinding Lights,5.3
1,Shape of You,4.8


In [ ]:
# RUN-TOGETHER
# Filter on text, and combine two conditions with &
# (parentheses required!).
pop = songs[songs['genre'] == 'Pop']
print('Pop titles:', list(pop['title']))

recent_pop = songs[(songs['genre'] == 'Pop') & (songs['year'] >= 2020)]
recent_pop[['title', 'year']]

Pop titles: ['Shape of You', 'Perfect', 'Die With a Smile']


,title,year
7,Die With a Smile,2024


### Activity

The previous example used `&` (and). Now use `|` (or), and predict before you run.

Find songs that are *either* released before 2005 *or* have more than 4.5 billion streams. Before running, guess how many rows you will get.


In [ ]:
# Activity code
# Before 2005 OR more than 4.5 billion -- predict the count first!
picks = _____  # songs[(songs['year'] < 2005) ____ (songs['streams_billions'] > 4.5)]
print('count:', picks.shape[0])
picks[['title', 'year', 'streams_billions']]

### 3.4 Sorting

`df.sort_values('column')` returns a new DataFrame ordered by that column. Add `ascending=False` for largest-first.

In [ ]:
# RUN-TOGETHER
# Most-streamed first.
top = songs.sort_values('streams_billions', ascending=False)
print(top[['title', 'streams_billions']].head())

# Oldest songs first (ascending is the default).
print(songs.sort_values('year')[['title', 'year']].head())

             title  streams_billions
0  Blinding Lights               5.3
1     Shape of You               4.8
2          Perfect               3.9
3         Believer               3.8
4       Heat Waves               3.7
                title  year
10  Bohemian Rhapsody  1975
5              Yellow  2000
11     Mr. Brightside  2004
6             Riptide  2013
8   Take Me to Church  2013


#### Example

a. Sort by two keys at once. Order by `genre` (A-Z), and *within* each genre by `streams_billions` (highest first). Pass a list of columns and a list of directions to `ascending` (e.g. `ascending=[True, False]`).

b. Using sort + position, print the `title` of the *3rd-most-streamed* song overall.

In [ ]:
# Exammple
# a. genre A-Z, then streams high-to-low within each genre
by_genre = songs.sort_values(['genre', 'streams_billions'], ascending=[____, ____])
print(by_genre[['genre', 'title', 'streams_billions']])

# b. 3rd-most-streamed overall: sort high-to-low, then take position 2 (0, 1, 2, ...)
third = songs.sort_values('streams_billions', ascending=False).iloc[____]
print('3rd most-streamed:', third['title'])

### 3.5 Creating new columns

We often need a value the raw data does not contain, computed from columns we do have. Assigning
to a *new* column name creates it. Column math is element-wise (thanks to NumPy underneath), so no loop is needed.

In [ ]:
# RUN-TOGETHER
# Age of each song, in years.
songs['age_years'] = 2026 - songs['year']

# A rate: average billions of streams PER year on the platform.
songs['streams_per_year'] = (songs['streams_billions'] / songs['age_years']).round(2)

# A boolean flag column.
songs['is_recent'] = songs['year'] >= 2020

songs[['title', 'year', 'age_years', 'streams_billions', 'streams_per_year', 'is_recent']].head()

,title,year,age_years,streams_billions,streams_per_year,is_recent
0,Blinding Lights,2019,7,5.3,0.76,False
1,Shape of You,2017,9,4.8,0.53,False
2,Perfect,2017,9,3.9,0.43,False
3,Believer,2017,9,3.8,0.42,False
4,Heat Waves,2020,6,3.7,0.62,True


#### Example

Build a column to *answer a question*, then interpret a rate.

a. Create `gap_to_leader` = how far each song trails the top song (the **max** streams minus that song's streams). Then filter to the songs *within 1.0 billion* of the leader.

b. Sort by `streams_per_year` (highest first). In a comment, name the song on top and explain how a *2024* release can out-rank songs with far more *total* streams.

In [ ]:
# Example code
# a. distance behind the most-streamed song
leader = songs['streams_billions'].____()          # the max value
songs['gap_to_leader'] = (leader - songs['streams_billions']).round(1)
close = songs[songs['gap_to_leader'] ____ 1.0]     # within 1 billion of the top
display(close[['title', 'streams_billions', 'gap_to_leader']])

# b. fastest accumulators: streams per year, highest first
songs['streams_per_year'] = (songs['streams_billions'] / (2026 - songs['year'])).round(2)
display(songs.sort_values('streams_per_year', ascending=False)[['title', 'year', 'streams_per_year']].head(3))
# Which song leads per year, and why can a 2024 song beat older giants?
# your answer: ____

### 3.6 Summaries and descriptive statistics

A few workhorses:

- `df['col'].mean()`, `.median()`, `.min()`, `.max()`, `.sum()` -> one number about a column.
- `df['col'].value_counts()` -> counts of each category in a text column.
- `df.groupby('group_col')['value_col'].mean()` -> one summary *per group*.
- `df.describe()` -> count, mean, std, min, quartiles, max for every numeric column at once.

In [ ]:
# RUN-TOGETHER
print('mean streams (billions):  ', round(songs['streams_billions'].mean(), 2))
print('median streams (billions):', songs['streams_billions'].median())
print('newest release year:', songs['year'].max())
print()

# How many songs per genre? (a category count)
print('songs per genre:')
print(songs['genre'].value_counts())

mean streams (billions):   3.73
median streams (billions): 3.55
newest release year: 2024

songs per genre:
genre
Pop                 3
Rock                2
Synth-pop           1
Pop rock            1
Indie pop           1
Alternative rock    1
Indie folk          1
Indie rock          1
R&B                 1
Name: count, dtype: int64


In [ ]:
# RUN-TOGETHER
# groupby: average streams per genre (one number per genre).
avg_by_genre = songs.groupby('genre')['streams_billions'].mean().round(2)
print(avg_by_genre.sort_values(ascending=False))

genre
Synth-pop           5.30
Pop                 4.07
Pop rock            3.80
Indie pop           3.70
Alternative rock    3.60
Indie folk          3.50
Indie rock          3.40
R&B                 3.20
Rock                3.05
Name: streams_billions, dtype: float64


# Example - Cars

Suppose we have vehicle data from different countries. Each observation corresponds to a country and the columns give information about the number of vehicles per capita, whether people drive left or right, and so on.



In [ ]:
# Example - Cars

# Build up a dictionary
# First, some data (in lists)
names = ['United States', 'Australia', 'Japan', 'India', 'Russia', 'Morocco', 'Egypt']
dr =  [True, False, False, False, True, True, True]
cpc = [809, 731, 588, 18, 200, 70, 45]

# Create dictionary my_dict with three key:value pairs: my_dict
my_dict = {'country':names, 'drives_right':dr, 'cars_per_cap':cpc}

# Build a DataFrame cars from my_dict: cars
cars = pd.DataFrame(my_dict)

# Print cars
print(cars)

## Add row labels
# Definition of row_labels
row_labels = ['US', 'AUS', 'JPN', 'IN', 'RU', 'MOR', 'EG']

# Specify row labels of cars
cars.index = row_labels

print("")
print(cars)

         country  drives_right  cars_per_cap
0  United States          True           809
1      Australia         False           731
2          Japan         False           588
3          India         False            18
4         Russia          True           200
5        Morocco          True            70
6          Egypt          True            45

           country  drives_right  cars_per_cap
US   United States          True           809
AUS      Australia         False           731
JPN          Japan         False           588
IN           India         False            18
RU          Russia          True           200
MOR        Morocco          True            70
EG           Egypt          True            45


In [ ]:
# Example - Cars
from google.colab import drive
drive.mount('/content/drive/')

# Starting from a .csv file
file_path = '/content/drive/MyDrive/Google Drive/Google Drive/Courses/DSC210/Notes/06-dictionaries_pandas/cars.csv'
cars = pd.read_csv(file_path, index_col=0)



# Print out cars
print(cars) # Note the row names and first column


Mounted at /content/drive/
           country  drives_right  cars_per_cap
US   United States          True           809
AUS      Australia         False           731
JPN          Japan         False           588
IN           India         False            18
RU          Russia          True           200
MOR        Morocco          True            70
EG           Egypt          True            45


# Indexing and Selecting Data

There are several ways to index or select data from pandas DataFrame objects
- Column indexing with square brackets (like lists and arrays)
    - Single brackets `[]` with a variable name yield a Series object
    - Double brackets `[[]]` with a variable name yield a DataFrame object
- Row indexing methods
    - Single brackets `[]` with an integer value (or range) yield a DataFrame object (a slice)
    - `.loc` for label-based indexing
    - `.iloc` for integer-based indexing
    - `[]` vs `[[]]` behavior for `.loc` and `.iloc` methods is similar to column indexing
    - Can be extended to select rows *and* columns


In [ ]:
# Example - Cars
# Column Access
## Access by name
print("Accessing column by name (country) -- Series return:")
print(cars["country"]) # Note the print out and the last line
print(type(cars["country"])) # Pandas 'series' object, effectively a labeled 1D array

print("\n \n Accessing column by name (country) -- DataFrame return:")
print(cars[["country"]])
print(type(cars[["country"]])) # Pandas 'DataFrame' object



print("\n \n Selecting several columns:")
print(cars[["country", "drives_right"]])

# Row Access
## Access by index
# Print out first 3 observations
print("\n \n Row access by index:")
print(cars[0:3])

# Print out fourth, fifth and sixth observation
print(cars[3:6])



Accessing column by name (country) -- Series return:
US     United States
AUS        Australia
JPN            Japan
IN             India
RU            Russia
MOR          Morocco
EG             Egypt
Name: country, dtype: object
<class 'pandas.core.series.Series'>

 
 Accessing column by name (country) -- DataFrame return:
           country
US   United States
AUS      Australia
JPN          Japan
IN           India
RU          Russia
MOR        Morocco
EG           Egypt
<class 'pandas.core.frame.DataFrame'>

 
 Selecting several columns:
           country  drives_right
US   United States          True
AUS      Australia         False
JPN          Japan         False
IN           India         False
RU          Russia          True
MOR        Morocco          True
EG           Egypt          True

 
 Row access by index:
           country  drives_right  cars_per_cap
US   United States          True           809
AUS      Australia         False           731
JPN          Japan      

In [ ]:
# Example - Cars

## Using loc and iloc
print("Row access by name -- Series return:")
print(cars.loc["JPN"])
print(type(cars.loc["JPN"]))

print("\n \n Row access by name -- DataFrame return:")
print(cars.loc[["JPN"]])
print(type(cars.loc[["JPN"]]))

print("\n \n Multiple rows by name:")
print(cars.loc[["JPN", "AUS"]])

print("\n \n Row and column access by name(s):")
print(cars.loc[["MOR"], ["drives_right"]])

print(cars.loc[["RU", "MOR"], ["country", "drives_right"]])


print("\n \n All row access (:) with column name, Series return")
print(cars.loc[:, "drives_right"])

# Print out drives_right column as DataFrame
print("\n \n All row access (:) with column name(s), DataFrame return")
print(cars.loc[:, ["drives_right"]])

# Print out cars_per_cap and drives_right as DataFrame
print(cars.loc[:, ["cars_per_cap", "drives_right"]])

Row access by name -- Series return:
country         Japan
drives_right    False
cars_per_cap      588
Name: JPN, dtype: object
<class 'pandas.core.series.Series'>

 
 Row access by name -- DataFrame return:
    country  drives_right  cars_per_cap
JPN   Japan         False           588
<class 'pandas.core.frame.DataFrame'>

 
 Multiple rows by name:
       country  drives_right  cars_per_cap
JPN      Japan         False           588
AUS  Australia         False           731

 
 Row and column access by name(s):
     drives_right
MOR          True
     country  drives_right
RU    Russia          True
MOR  Morocco          True

 
 All row access (:) with column name, Series return
US      True
AUS    False
JPN    False
IN     False
RU      True
MOR     True
EG      True
Name: drives_right, dtype: bool

 
 All row access (:) with column name(s), DataFrame return
     drives_right
US           True
AUS         False
JPN         False
IN          False
RU           True
MOR          Tru

---
## 4. Guided Practice: esports club
---

### The scenario

You volunteer for the campus **esports club**, which is building a "**Featured Indie Games**"
board for the student center. A club member collected a fresh batch of games and handed you the
notes as a **list of records** (a list of dictionaries) -- exactly the messy-but-common format
real data arrives in.

Your job walks through a realistic mini-pipeline:

- **Part A.** Turn the raw records into a DataFrame and inspect it. *(dictionaries -> DataFrame)*
- **Part B.** Enrich it with a derived column and a **lookup dictionary**, then filter and sort.
- **Part C.** Summarize and write a one-sentence recommendation.

**Predict each output before you run it.**

> The `club_score` values below are the club's own made-up ratings (out of 10), included so we
> have a numeric column to sort and average. Titles, studios, years, and genres are factual.

#### Part A. From raw records to a table

Build a DataFrame from the list of records and get oriented.

In [ ]:
# Part A -- fill in the blanks
import pandas as pd

# Build up a dictionary
# First, some data (in lists)
title = ['Portal 2', 'Celeste', 'Hades', 'Overcooked! 2', 'Cuphead', "Baldur's Gate 3" ]
studio =  ['Valve', 'Maddy Makes Games', 'Supergiant Games', 'Ghost Town Games', 'Studio MDHR', 'Larian Studios']
year = [2011, 2018, 2020, 2018, 2017, 2023]
indy = [False, True, True, True, True, True]
# Create dictionary club_dict with four key:value pairs:
club_dict = {'title':title, 'studio':studio, 'year':year, 'is_indy': indy}

# 1. Build a DataFrame from the list of records
club = pd.DataFrame(club_dict)

# 2. Inspect: shape, then the first few rows.
print('shape:', club.____)
club.head()

             title             studio  year  is_indy
0         Portal 2              Valve  2011    False
1          Celeste  Maddy Makes Games  2018     True
2            Hades   Supergiant Games  2020     True
3    Overcooked! 2   Ghost Town Games  2018     True
4          Cuphead        Studio MDHR  2017     True
5  Baldur's Gate 3     Larian Studios  2023     True


#### Part B. Enrich with a derived column and a lookup dictionary

Now lets add two new columns:

1. A **derived column** `age_years`.
2. A **lookup dictionary** from title to the esport club's rating. We *loop over the `title` column* (visiting its values top to bottom, just like looping over a list), look each title up in the dictionary, collect the scores into a list, and attach that list as a new column.
3. Then filter to recent games and sort by the club's score.

In [ ]:
# Part B -- fill in the blanks

# 1. Derived column: how old is each game in 2026?
club['age_years'] = 2026 - club[____]

# 2. A lookup dictionary: title -> club rating (out of 10, made up for this exercise).
club_score = {
    'Portal 2': 9,
    'Celeste': 10,
    'Hades': 10,
    'Overcooked! 2': 8,
    'Cuphead': 9,
    "Baldur's Gate 3": 10,
}

# Look each title up in the dictionary, collect the scores, and attach them as a new column.
scores = []
for title in club['title']:          # looping a column visits its values, like a list
    scores.append(club_score[____])  # look up THIS title's score in the dictionary
station['club_score'] = scores


# 3. Filter to games from 2018 or later, then sort by club_score (highest first).
recent = club[club['year'] >= ____]
recent.sort_values(____, ascending=False)

#### Part C. Summarize and recommend

Answer the club's questions, then write your recommendation as a comment.

1. What is the **average** `club_score` across all games?
2. How many games are there **per genre**?
3. Which single game would you feature first? Use the sorted table from Part B to justify it.

In [ ]:
# Part C -- fill in the blanks

# 1. Average club score across all games.
print('mean club score:', round(club['club_score'].____(), 2))

# 2. Games per genre.
print()
print(club[____].value_counts())

# 3. Top pick: highest club_score, tie broken by newest year.
top_pick = club.sort_values(['club_score', 'year'], ascending=False).____(1)
print()
print(top_pick[['title', 'year', 'genre', 'club_score']])

# Our recommendation (write one sentence as a comment):
#